In [1]:
import os
from pyspark.sql import SparkSession, functions as F

# Имя каталога
catalog = "lk"

# Доступ к minio
access_key = os.getenv("MINIO_ROOT_USER", "minioadmin")
secret_key = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
warehouse = os.getenv("LAKEKEEPER_WAREHOUSE", "mydatalab")

#Настройка каталога в Spark
spark = (
    SparkSession.builder.appName("lakekeeper-iceberg-demo")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{catalog}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{catalog}.type", "rest")
    .config(f"spark.sql.catalog.{catalog}.uri", "http://127.0.0.1:8181/catalog")
    .config(f"spark.sql.catalog.{catalog}.warehouse", warehouse)
    .config(f"spark.sql.catalog.{catalog}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{catalog}.s3.endpoint", "http://127.0.0.1:9000")
    .config(f"spark.sql.catalog.{catalog}.s3.path-style-access", "true")
    .config(f"spark.sql.catalog.{catalog}.s3.access-key-id", access_key)
    .config(f"spark.sql.catalog.{catalog}.s3.secret-access-key", secret_key)
    .config(f"spark.sql.legacy.parquet.nanosAsLong","true") #Добавили для поддержки Timestamp из паркета
    .config("spark.sql.defaultCatalog", catalog)
    .getOrCreate()
)

#Уровень логирования
spark.sparkContext.setLogLevel("WARN")

In [13]:
print(f"stage.users:{spark.sql("""SELECT count(1) || '|' || sum(file_size_in_bytes) FROM stage.users.all_data_files""").collect()[0][0]}")
print(f"stage.payments:{spark.sql("""SELECT count(1) || '|' || sum(file_size_in_bytes) FROM stage.payments.all_data_files""").collect()[0][0]}")
print(f"stage.trips:{spark.sql("""SELECT count(1) || '|' || sum(file_size_in_bytes) FROM stage.trips.all_data_files""").collect()[0][0]}")
print(f"stage.events:{spark.sql("""SELECT count(1) || '|' || sum(file_size_in_bytes) FROM stage.events.all_data_files""").collect()[0][0]}")
print(f"dds.users:{spark.sql("""SELECT count(1) || '|' || sum(file_size_in_bytes) FROM dds.users.all_data_files""").collect()[0][0]}")
print(f"dds.payments:{spark.sql("""SELECT count(1) || '|' || sum(file_size_in_bytes) FROM dds.payments.all_data_files""").collect()[0][0]}")
print(f"dds.trips:{spark.sql("""SELECT count(1) || '|' || sum(file_size_in_bytes) FROM dds.trips.all_data_files""").collect()[0][0]}")
print(f"dds.events:{spark.sql("""SELECT count(1) || '|' || sum(file_size_in_bytes) FROM dds.events.all_data_files""").collect()[0][0]}")
print(f"mart.user_trips:{spark.sql("""SELECT count(1) || '|' || sum(file_size_in_bytes) FROM mart.user_trips.all_data_files""").collect()[0][0]}")

stage.users:2|63815
stage.payments:2|967797
stage.trips:2|4579398
stage.events:2|3892205
dds.users:2|70356
dds.payments:3|1452213
dds.trips:5|8873866
dds.events:2|3892896
mart.user_trips:1|821413
